In [2]:
import os, yaml, json


configs/few_shot/xxx_few_shot.yaml


In [1]:
import os
import yaml

config_dir = '/root/wj/EZ_CLIP/configs/few_shot'
dataset = ['HARDVS', 'PAF', 'DVS128Gesture', 'SeAct']
config_file_templates = ['{}_few_shot.yaml']
few_shot = ['2', '4', '8_shot', '16_shot']

for d in dataset:
    for config_template in config_file_templates:
        config_file = os.path.join(config_dir, config_template.format(d))
        print(config_file)
        # Read existing YAML content
        if os.path.exists(config_file):
            with open(config_file, 'r') as infile:
                config_data = yaml.safe_load(infile) or {}
        else:
            config_file_base = os.path.join(config_dir, 'basic_few_shot.yaml')
            with open(config_file_base, 'r') as infile:
                config_data = yaml.safe_load(infile) or {}
        # base_to_novel & create relative path file
        few_shot_txt = f'{d}_few_shot_train.txt'
        config_data['data']['train_list'] = f'dataset_splits/{d}/few_shot/' + few_shot_txt
        config_data['data']['val_list'] = f'dataset_splits/{d}/Zero-shot/val.txt'
        config_data['data']['gpt_discription'] = f'GPT_discription/{d}_gpt_Class_discription_new.csv'
        config_data['data']['label_list'] = f'lists/{d}_labels.csv'
        # basic setting
        config_data['network']['type'] = 'few_shot' 
        config_data['weight_save_dir'] = '/root/autodl-tmp/SAMPLE'
        config_data['training_name'] = f'{few_shot_txt.replace(".txt", "")}'
        config_data['network']['sim_header'] = 'Transf'
        config_data['mm_prompt']['CTX_INIT'] = 'a human action of'
        config_data['pretrain'] = None
        config_data['resume'] = None
        config_data['T_Adapter'] = False
        config_data['data']['dataset'] = d
        config_data['solver']['epochs'] = 100
        # time prompt
        config_data['prompt']['use']= True
        config_data['prompt']['DEEP']= True
        # mm prompt
        config_data['mm_prompt']['use'] = True
        config_data['mm_prompt']['N_CTX']= 2
        config_data['mm_prompt']['PROMPT_DEPTH']= 9

        # Print log directory
        logdir = os.path.join(
            config_data["weight_save_dir"],
            config_data["network"]["type"],
            config_data["network"].get("arch", "unknown_arch"),
            config_data["data"].get("dataset", d),
            config_data['training_name']
        )

        # 创建必要的文件和目录
        few_shot_dir = f'/root/wj/EZ_CLIP/dataset_splits/{d}/few_shot'
        os.makedirs(few_shot_dir, exist_ok=True)
        
        files_to_create = [
            f'{d}_few_shot_train_2.txt',
            f'{d}_few_shot_train_4.txt',
            f'{d}_few_shot_train_8.txt',
            f'{d}_few_shot_train_16.txt',
            f'{d}_few_shot_val.txt'
        ]
        
        for file in files_to_create:
            file_path = os.path.join(few_shot_dir, file)
            if not os.path.exists(file_path):
                with open(file_path, 'w') as f:
                    pass  # 创建空文件
        
        # 创建GPT描述文件
        gpt_description_dir = '/root/wj/EZ_CLIP/GPT_discription'
        os.makedirs(gpt_description_dir, exist_ok=True)
        gpt_description_file = f'{d}_gpt_Class_discription_new.csv'
        gpt_description_path = os.path.join(gpt_description_dir, gpt_description_file)
        if not os.path.exists(gpt_description_path):
            with open(gpt_description_path, 'w') as f:
                pass  # 创建空文件
        
        print(f"已为数据集 {d} 创建所需的文件和目录")
        
        # Write back to YAML file
        with open(config_file, 'w') as outfile:
            yaml.dump(config_data, outfile, default_flow_style=False)

/root/wj/EZ_CLIP/configs/few_shot/HARDVS_few_shot.yaml
已为数据集 HARDVS 创建所需的文件和目录
/root/wj/EZ_CLIP/configs/few_shot/PAF_few_shot.yaml
已为数据集 PAF 创建所需的文件和目录
/root/wj/EZ_CLIP/configs/few_shot/DVS128Gesture_few_shot.yaml
已为数据集 DVS128Gesture 创建所需的文件和目录
/root/wj/EZ_CLIP/configs/few_shot/SeAct_few_shot.yaml
已为数据集 SeAct 创建所需的文件和目录


dataset_split/xxx/few_shot/xxx_few_shot_train_2.txt 
dataset_split/xxx/few_shot/xxx_few_shot_train_4.txt 
dataset_split/xxx/few_shot/xxx_few_shot_train_8.txt 
dataset_split/xxx/few_shot/xxx_few_shot_train_16.txt 
dataset_split/xxx/few_shot/xxx_few_shot_val.txt

In [1]:
# 1. 创建数据集的few_shot_train_2.txt, few_shot_train_4.txt, few_shot_train_8.txt, few_shot_train_16.txt, few_shot_val.txt
# 方法：从训练集中, 每个标签下面的样本中分别随机抽取2, 4, 8, 16个样本作为few_shot_train_2.txt, few_shot_train_4.txt, few_shot_train_8.txt, few_shot_train_16.txt，剩余的作为few_shot_val.txt
# 训练集：dataset_split/xxx/Zero-shot/train.txt
# 格式：path, num_segments, label
# e.g. /root/autodl-tmp/HARDVS_Sampled_EZCLIP/action_001/dvSave-2021_08_20_16_15_22 8 0
import random, os

def create_few_shot_files(train_file, few_shot_dir, num_samples_list):
    # 读取训练集文件
    with open(train_file, 'r') as f:
        lines = f.readlines()
    
    # 按标签分类样本
    samples_by_label = {}
    for line in lines:
        path, num_segments, label = line.strip().split()
        if label not in samples_by_label:
            samples_by_label[label] = []
        samples_by_label[label].append(line)
    
    # 创建few_shot文件
    for num_samples in num_samples_list:
        few_shot_file = os.path.join(few_shot_dir, f'{d}_few_shot_train_{num_samples}.txt')
        with open(few_shot_file, 'w') as f:
            for label, samples in samples_by_label.items():
                selected_samples = random.sample(samples, min(num_samples, len(samples)))
                f.writelines(selected_samples)


# 示例调用
dataset = ['HARDVS', 'PAF', 'DVS128Gesture', 'SeAct']
for d in dataset:
    train_file = f'/root/wj/EZ_CLIP/dataset_splits/{d}/Zero-shot/train.txt'
    few_shot_dir = f'/root/wj/EZ_CLIP/dataset_splits/{d}/few_shot'
    num_samples_list = [2, 4, 8, 16]
    create_few_shot_files(train_file, few_shot_dir, num_samples_list)




